# Handwritten Character Recognition using CNNs
### CodeAlpha Deep Learning Internship Project

**Objective:**
Develop a Convolutional Neural Network (CNN) using TensorFlow/Keras that can recognize handwritten digits and characters from the EMNIST Balanced dataset (47 classes) or MNIST (10 classes). This notebook demonstrates:
1. Libraries import & setup
2. Data loading and preprocessing
3. Exploratory Data Analysis (EDA) & Visualizations
4. CNN Model Architecture design
5. Model training and validation
6. Detailed model performance evaluation (Classification Report, Confusion Matrix Heatmap)

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# Set plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12
print("TensorFlow version:", tf.__version__)

## 2. Load and Preprocess Dataset
We use the **EMNIST Balanced** dataset which contains 112,800 training samples and 18,800 test samples of 47 balanced classes (0-9, A-Z, and some lowercase letters).

In [ ]:
import emnist

# EMNIST Balanced classes mapping
class_names = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
    'a', 'b', 'd', 'e', 'f', 'g', 'h', 'n', 'q', 'r', 't'
]

# Load training and testing samples
x_train, y_train = emnist.extract_training_samples('balanced')
x_test, y_test = emnist.extract_test_samples('balanced')

print(f"Training data shape: {x_train.shape}, labels: {y_train.shape}")
print(f"Testing data shape: {x_test.shape}, labels: {y_test.shape}")

### Data Preprocessing
- Normalizing pixel values to the range `[0, 1]`.
- Reshaping image dimensions from `(28, 28)` to `(28, 28, 1)` for CNN compatibility.
- One-hot encoding labels using Keras utility `to_categorical`.

In [ ]:
# Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Reshape for CNN
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# One-hot encode
num_classes = len(class_names)
y_train_encoded = to_categorical(y_train, num_classes=num_classes)
y_test_encoded = to_categorical(y_test, num_classes=num_classes)

print(f"Reshaped Train images: {x_train.shape}, encoded labels: {y_train_encoded.shape}")
print(f"Reshaped Test images: {x_test.shape}, encoded labels: {y_test_encoded.shape}")

## 3. Exploratory Data Analysis (EDA)
Let's visualize some samples and check the class distribution to ensure there's no imbalance.

### Display Random Samples

In [ ]:
plt.figure(figsize=(10, 10))
indices = np.random.randint(0, len(x_train), size=25)

for idx, i in enumerate(indices):
    plt.subplot(5, 5, idx + 1)
    # EMNIST images are loaded transposed/rotated in some libraries, but the emnist package outputs them correctly orientated.
    # If they appear rotated, we can use np.rot90 or transpose them. Let's check how they look.
    plt.imshow(x_train[i].squeeze(), cmap='gray')
    plt.title(f"Label: {class_names[y_train[i]]}")
    plt.axis('off')

plt.tight_layout()
plt.show()

### Class Distribution

In [ ]:
# Verify dataset balance
unique, counts = np.unique(y_train, return_counts=True)
class_counts = dict(zip([class_names[u] for u in unique], counts))

plt.figure(figsize=(14, 5))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()), palette='viridis')
plt.title('EMNIST Balanced Training Set - Class Distribution')
plt.xlabel('Class Label')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

print(f"Min count per class: {min(counts)}, Max count per class: {max(counts)}")

## 4. Build CNN Architecture using Keras
We design a Convolutional Neural Network (CNN) architecture with:
- **Convolution Layers**: For feature extraction (using ReLU activations).
- **MaxPooling Layers**: For spatial downsampling.
- **Dropout Layers**: For regularization to prevent overfitting.
- **Dense Layer**: For learning combinations of high-level features.
- **Softmax Output**: For multi-class classification probability distribution.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.regularizers import l2

def build_model(input_shape=(28, 28, 1), num_classes=47):
    model = Sequential([
        Input(shape=input_shape),
        
        # Convolution block 1
        Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
        Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # Convolution block 2
        Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # Flattening & FC Layers
        Flatten(),
        Dense(256, activation='relu', kernel_regularizer=l2(0.0001)),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(num_classes=num_classes)
model.summary()

## 5. Train the Model
We train the model using early stopping to prevent overfitting if the validation loss stops improving.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train for a few epochs (adjust epochs for better accuracy)
epochs = 8
batch_size = 128

history = model.fit(
    x_train, y_train_encoded,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.15,
    callbacks=[early_stopping],
    verbose=1
)

### Save the Model

In [ ]:
os.makedirs('../models', exist_ok=True)
model.save('../models/emnist_model.keras')
print("Model saved to '../models/emnist_model.keras'")

### Visualizing Learning Curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy curves
ax[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
ax[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='o')
ax[0].set_title('Training vs Validation Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

# Loss curves
ax[1].plot(history.history['loss'], label='Train Loss', marker='o')
ax[1].plot(history.history['val_loss'], label='Val Loss', marker='o')
ax[1].set_title('Training vs Validation Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.show()

## 6. Evaluate Model Performance
We test the final model on our separate test set, calculating Accuracy, Precision, Recall, and F1-score.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_encoded, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Predict
y_pred_prob = model.predict(x_test, batch_size=128, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

# Compute classification metrics
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names))

### Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('EMNIST Balanced Confusion Matrix Heatmap')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()